# Argumentation basée sur les valeurs (VAF, Bench-Capon 2003)

[← Dung_AF_Semantics](Argumentation-03-Dung-AF-Semantics-Python.ipynb) | [↑ ArgumentAnalysis](README.md) | [Ranking_Semantics →](Argumentation-03c-Ranking-Semantics-Python.ipynb) | [jumeau Rust](Argumentation-03b-Value-Based-AF-Rust.ipynb) | [jumeau Python](Argumentation-03b-Value-Based-AF-Python.ipynb)

## Pourquoi ce notebook

Le notebook [Argumentation-03-Dung-AF-Semantics-Python](Argumentation-03-Dung-AF-Semantics-Python.ipynb) reconstruit l'argumentation *abstraite* de Dung (1995) : des arguments s'attaquent, des sémantiques (grounded / preferred / stable) décident lesquels survivent. Mais dans ce cadre, **toutes les attaques réussissent** — un argument qui en attaque un autre le défait, un point c'est tout.

La réalité du débat est différente. Considérez : « *Il faut réduire les impôts pour relancer la consommation* » (argument *A*, valeur **croissance**) attaque « *Il faut maintenir les impôts pour financer les services publics* » (argument *B*, valeur **solidarité**). L'attaque de *A* sur *B* doit-elle réussir ? Cela dépend de ce que **valorise l'auditoire** : un auditoire qui préfère la croissance à la solidarité laissera *A* défaire *B* ; un auditoire inverse laissera *B* résister à *A*. **Le même graphe d'attaques produit des conclusions différentes selon l'audience.**

C'est le saut conceptuel de Trevor Bench-Capon dans *« A Value-Based Argumentation Framework »* (2003) : enrichir le cadre de Dung par des **valeurs** (ce que chaque argument promeut) et une **audience** (un ordre de préférence sur ces valeurs). Une attaque ne défait sa cible que si la valeur de l'attaquant est au moins aussi préférée que celle de l'attaqué, aux yeux de l'audience.

Ce notebook reconstruit le cadre VAF de zéro en **C# pur (BCL .NET, sans dépendance externe)**, à la suite de `Dung_AF_Semantics` dont il reprend les primitives. Objectif pédagogique : voir concrètement comment **trois audiences différentes** sur un même graphe d'attaques en cycle produisent **trois conclusions différentes** — et pourquoi ce phénomène connecte l'argumentation au choix social (préférences agrégées).


## Le cadre VAF — définition

Un **Value-based Argumentation Framework** (VAF) est un quintuplet $\langle \mathcal{A}, \mathcal{R}, V, \eta, \succcurlyeq_a \rangle$ où :

- $\mathcal{A}$ est un ensemble d'arguments (comme chez Dung) ;
- $\mathcal{R} \subseteq \mathcal{A} \times \mathcal{A}$ est la relation d'attaque (comme chez Dung) ;
- $V$ est un ensemble de **valeurs** ;
- $\eta : \mathcal{A} \to V$ associe à chaque argument la valeur qu'il promeut ;
- $\succcurlyeq_a$ est l'**audience** : un ordre total (préférence) sur $V$.

**Défaite dépendante de l'audience.** L'argument $x$ *défait* $y$ (l'attaque $x \to y$ réussit) si et seulement si :

$$\eta(x) \succcurlyeq_a \eta(y)$$

Sinon, $y$ **résiste** à $x$ : l'attaque échoue, $y$ n'est pas défait par $x$.

Une fois cette relation de défaite audience-dépendante fixée, on réapplique les sémantiques de Dung (grounded, preferred, stable) dessus. Le résultat **dépend de l'audience** : changer l'ordre de préférence sur $V$ change qui défait qui, donc change les extensions.


In [1]:
// Primitives de Dung (auto-contenues, comme Dung_AF_Semantics).
using System;
using System.Collections.Generic;
using System.Linq;
using System.Numerics;

/// <summary>
/// Cadre d'argumentation abstrait : arguments + relation d'attaque.
/// Ici `attacks` = relation de DEFAITE deja resolue (cf. AfFromAudience plus bas).
/// On construit un AF a partir d'un VAF + une audience, puis on lui applique
/// les semantiques de Dung.
/// </summary>
public class AF
{
    public HashSet<string> Args { get; }

    // couples (attaquant, attaque)
    public HashSet<(string, string)> Attacks { get; }

    public AF(IEnumerable<string> args, IEnumerable<(string, string)> attacks)
    {
        Args = new HashSet<string>(args);
        Attacks = new HashSet<(string, string)>(attacks);
    }

    public IEnumerable<string> Attackers(string x) =>
        Attacks.Where(t => t.Item2 == x).Select(t => t.Item1);

    // Rendu stable d'un ensemble d'arguments (ordre lexicographique).
    public static string Format(IEnumerable<string> items)
    {
        var xs = items.OrderBy(s => s, StringComparer.Ordinal).ToList();
        return "{" + string.Join(", ", xs) + "}";
    }

    // Rendu stable d'un ensemble d'attaques (attaquant->cible).
    public static string FormatAttacks(IEnumerable<(string, string)> attacks)
    {
        var xs = attacks
            .OrderBy(t => t.Item1, StringComparer.Ordinal)
            .ThenBy(t => t.Item2, StringComparer.Ordinal)
            .Select(t => t.Item1 + "->" + t.Item2);
        return "[" + string.Join(", ", xs) + "]";
    }

    public override string ToString()
    {
        var att = FormatAttacks(Attacks);
        var args = string.Join(", ", Args.OrderBy(a => a, StringComparer.Ordinal));
        return $"AF(args=[{args}], attacks={att})";
    }
}

/// <summary>Semantiques de Dung appliquees a un cadre resolu.</summary>
public static class DungSemantics
{
    /// True si S defait x (au moins un membre de S attaque x).
    public static bool Defeats(AF af, IEnumerable<string> s, string x) =>
        s.Any(a => af.Attacks.Contains((a, x)));

    /// True si S defend x : tout attaquant de x est defait par S.
    public static bool Defends(AF af, IEnumerable<string> s, string x) =>
        af.Attackers(x).All(b => Defeats(af, s, b));

    /// Extension grounded : plus petit point fixe de la fonction de defense.
    public static HashSet<string> Grounded(AF af)
    {
        var e = new HashSet<string>();
        while (true)
        {
            var eNew = new HashSet<string>(e);
            foreach (var x in af.Args)
                if (Defends(af, e, x))
                    eNew.Add(x);
            if (eNew.SetEquals(e))
                return e;
            e = eNew;
        }
    }
}

/// <summary>
/// Enumeration des sous-ensembles (powerset), sous-ensemble vide compris.
/// L'ordre reproduit itertools.combinations : cardinalite croissante, puis
/// ordre lexicographique des indices, en conservant les occurrences repetees.
/// </summary>
public static class Sets
{
    private static IEnumerable<int[]> Combinations(int n, int r, int start)
    {
        if (r == 0)
        {
            yield return Array.Empty<int>();
            yield break;
        }

        for (int i = start; i <= n - r; i++)
        {
            foreach (var tail in Combinations(n, r - 1, i + 1))
            {
                var result = new int[r];
                result[0] = i;
                Array.Copy(tail, 0, result, 1, tail.Length);
                yield return result;
            }
        }
    }

    public static IEnumerable<List<string>> PowerSet(IEnumerable<string> items)
    {
        // Comme itertools.combinations, on conserve l'ordre et les occurrences
        // de l'entree : pas de Distinct() ni de tri preprocesseur.
        var xs = items.ToList();
        int n = xs.Count;

        for (int r = 0; r <= n; r++)
        {
            foreach (var indices in Combinations(n, r, 0))
            {
                yield return indices.Select(i => xs[i]).ToList();
            }
        }
    }

    /// <summary>
    /// Nombre exact de sous-ensembles, calcule avec System.Numerics.BigInteger.
    /// Avec n positions (occurrences repetees conservees), le nombre est 2^n.
    /// </summary>
    public static BigInteger PowerSetCount(IEnumerable<string> items)
    {
        int n = items.Count();
        return BigInteger.Pow(2, n);
    }
}

/// <summary>Controles executables : un ecart interrompt l'execution du carnet.</summary>
public static class Checks
{
    public static void Require(bool condition, string message)
    {
        if (!condition)
            throw new InvalidOperationException("ECHEC : " + message);
        Console.WriteLine("  CONFORME : " + message);
    }
}

Console.WriteLine("Primitives de Dung chargees. Smoke test:");
var afTest = new AF(new[] { "a", "b" }, new[] { ("a", "b") });
Console.WriteLine($"  af_test = {afTest}");
Console.WriteLine($"  grounded(af_test) = {AF.Format(DungSemantics.Grounded(afTest))}  // a defendu, b defait par a");

// --- Assertions executables : primitives de Dung ---
Checks.Require(DungSemantics.Grounded(afTest).SetEquals(new[] { "a" }),
    "grounded({a,b} ; a->b) == {a}");
Checks.Require(afTest.Attackers("b").OrderBy(s => s, StringComparer.Ordinal).SequenceEqual(new[] { "a" }),
    "attaquants de b == {a}");
Checks.Require(!afTest.Attackers("a").Any(), "a n'a aucun attaquant");
Checks.Require(DungSemantics.Defeats(afTest, new[] { "a" }, "b"), "a defait b");
Checks.Require(!DungSemantics.Defends(afTest, new[] { "a" }, "b"), "a ne defend pas b : il l'attaque");

// --- Assertions executables : powerset (equivalent de la primitive Python) ---
var powerset = Sets.PowerSet(new[] { "a", "b" }).ToList();
var powersetCount = Sets.PowerSetCount(new[] { "a", "b" });
Console.WriteLine($"  powerset({{a,b}}) = {string.Join(" | ", powerset.Select(s => "{" + string.Join(", ", s) + "}"))}");
Checks.Require(powersetCount == new BigInteger(4), "powerset de 2 elements : 4 sous-ensembles");
Checks.Require(powerset.Any(s => s.Count == 0), "le powerset contient le sous-ensemble vide");
Checks.Require(powerset.Any(s => s.SequenceEqual(new[] { "a", "b" })), "le powerset contient {a,b}");
Checks.Require(Sets.PowerSetCount(new[] { "a", "a", "b" }) == new BigInteger(8),
    "le powerset enumere toutes les positions, y compris les occurrences repetees");

var powerset3 = Sets.PowerSet(new[] { "a", "b", "c" }).ToList();
var powerset3Count = Sets.PowerSetCount(new[] { "a", "b", "c" });
var powerset3Texte = string.Join(" | ", powerset3.Select(s => "{" + string.Join(", ", s) + "}"));
Console.WriteLine($"  powerset({{a,b,c}}) = {powerset3Texte}");
Checks.Require(powerset3Texte == "{} | {a} | {b} | {c} | {a, b} | {a, c} | {b, c} | {a, b, c}" &&
              powerset3Count == new BigInteger(8),
    "l'ordre du powerset sur 3 elements reproduit itertools.combinations");

Primitives de Dung chargees. Smoke test:


  af_test = AF(args=[a, b], attacks=[a->b])


  grounded(af_test) = {a}  // a defendu, b defait par a


  CONFORME : grounded({a,b} ; a->b) == {a}


  CONFORME : attaquants de b == {a}


  CONFORME : a n'a aucun attaquant


... (5 outputs left out)

  CONFORME : le powerset contient {a,b}


  CONFORME : le powerset enumere toutes les positions, y compris les occurrences repetees


  powerset({a,b,c}) = {} | {a} | {b} | {c} | {a, b} | {a, c} | {b, c} | {a, b, c}


  CONFORME : l'ordre du powerset sur 3 elements reproduit itertools.combinations


In [2]:
// --- Le cadre VAF (Bench-Capon 2003) ---
using System;
using System.Collections.Generic;
using System.Linq;

/// <summary>
/// Value-based Argumentation Framework.
/// Attributs : args (arguments), attacks (relation d'attaque, COMME chez Dung),
/// values (arg -> valeur, eta), value_set (l'ensemble des valeurs V).
/// </summary>
public class VAF
{
    public HashSet<string> Args { get; }

    public HashSet<(string, string)> Attacks { get; }

    // arg -> valeur
    public Dictionary<string, string> Values { get; }

    public HashSet<string> ValueSet { get; }

    public VAF(IEnumerable<string> args, IEnumerable<(string, string)> attacks, IDictionary<string, string> values)
    {
        Args = new HashSet<string>(args);
        Attacks = new HashSet<(string, string)>(attacks);
        Values = new Dictionary<string, string>(values);
        ValueSet = new HashSet<string>(Values.Values);
    }

    public string ValueOf(string x) => Values[x];

    public override string ToString()
    {
        var v = string.Join(", ", Values
            .OrderBy(kv => kv.Key, StringComparer.Ordinal)
            .Select(kv => kv.Key + ":" + kv.Value));
        var att = AF.FormatAttacks(Attacks);
        var args = string.Join(", ", Args.OrderBy(a => a, StringComparer.Ordinal));
        return $"VAF(args=[{args}], attacks={att}, values={{{v}}})";
    }
}

/// <summary>Regle de defaite du VAF, dependante de l'audience.</summary>
public static class VafSemantics
{
    /// Rang d'une valeur v dans l'audience (0 = la plus preferee).
    /// `audience` = liste ordonnee des valeurs, de la plus preferee a la moins preferee.
    /// Plus le rang est PETIT, plus la valeur est preferee.
    public static int AudienceRank(IList<string> audience, string v)
    {
        int rank = audience.IndexOf(v);
        if (rank < 0)
            throw new ArgumentException($"La valeur '{v}' ne figure pas dans l'audience.");
        return rank;
    }

    /// Verifie qu'une audience est une permutation complete des valeurs du VAF.
    public static void ValidateAudience(VAF vaf, IList<string> audience)
    {
        if (audience == null)
            throw new ArgumentException("L'audience ne peut pas etre nulle.");
        if (audience.Count != vaf.ValueSet.Count || audience.Distinct().Count() != audience.Count)
            throw new ArgumentException("L'audience doit contenir chaque valeur du VAF exactement une fois.");
        if (!vaf.ValueSet.All(v => audience.Contains(v)))
            throw new ArgumentException("L'audience contient une valeur inconnue ou ne contient pas toutes les valeurs du VAF.");
    }

    /// True si l'attaque x->y reussit sous cette audience.
    /// L'attaque reussit ssi value(x) est au moins aussi preferee que value(y).
    /// On compare les RANGS : rang plus petit = plus prefere.
    public static bool DefeatsUnderAudience(VAF vaf, IList<string> audience, string x, string y)
    {
        ValidateAudience(vaf, audience);
        if (!vaf.Attacks.Contains((x, y)))
            return false;
        return AudienceRank(audience, vaf.ValueOf(x)) <= AudienceRank(audience, vaf.ValueOf(y));
    }

    /// Construit le AF (Dung) correspondant a un VAF sous une audience donnee.
    /// On ne garde que les attaques qui REUSSISSENT sous l'audience. Les attaques
    /// qui echouent (valeur de l'attaquant moins preferee) disparaissent : la
    /// cible resiste.
    public static AF AfFromAudience(VAF vaf, IList<string> audience)
    {
        ValidateAudience(vaf, audience);
        var surviving = vaf.Attacks
            .Where(t => DefeatsUnderAudience(vaf, audience, t.Item1, t.Item2))
            .ToList();
        return new AF(vaf.Args, surviving);
    }
}

// Smoke test : un VAF minimal a 2 arguments, 1 valeur commune.
var vafTest = new VAF(
    new[] { "a", "b" },
    new[] { ("a", "b") },
    new Dictionary<string, string> { ["a"] = "v", ["b"] = "v" });
var audTest = new List<string> { "v" };
var afT = VafSemantics.AfFromAudience(vafTest, audTest);
Console.WriteLine($"VAF test: {vafTest}");
Console.WriteLine($"  audience [{string.Join(", ", audTest)}] -> AF survived attacks: {AF.FormatAttacks(afT.Attacks)}");
Console.WriteLine($"  grounded = {AF.Format(DungSemantics.Grounded(afT))}  // valeurs egales -> attaque reussit -> a defait b");

// --- Assertions executables : regle de defaite VAF ---
Checks.Require(VafSemantics.AudienceRank(audTest, "v") == 0, "rang de la valeur v dans l'audience triviale == 0");
Checks.Require(VafSemantics.DefeatsUnderAudience(vafTest, audTest, "a", "b"), "l'attaque a->b reussit (valeurs egales)");
Checks.Require(!VafSemantics.DefeatsUnderAudience(vafTest, audTest, "b", "a"), "b->a n'est pas une attaque : aucune defaite");
Checks.Require(AF.FormatAttacks(afT.Attacks) == "[a->b]", "attaques survivantes du VAF minimal == [a->b]");
Checks.Require(DungSemantics.Grounded(afT).SetEquals(new[] { "a" }), "grounded du VAF minimal == {a}");


VAF test: VAF(args=[a, b], attacks=[a->b], values={a:v, b:v})


  audience [v] -> AF survived attacks: [a->b]


  grounded = {a}  // valeurs egales -> attaque reussit -> a defait b


  CONFORME : rang de la valeur v dans l'audience triviale == 0


  CONFORME : l'attaque a->b reussit (valeurs egales)


  CONFORME : b->a n'est pas une attaque : aucune defaite


  CONFORME : attaques survivantes du VAF minimal == [a->b]


  CONFORME : grounded du VAF minimal == {a}


## Exemple canonique : le cycle à trois valeurs

Voici le moment pédagogique clé. Prenons trois arguments en **cycle** ($a \to b \to c \to a$), chacun promouvant une **valeur distincte** :

| Argument | Valeur promue |
|----------|---------------|
| $a$ | `verite` |
| $b$ | `confiance` |
| $c$ | `solidarite` |

Chez Dung (sans valeurs), un cycle de longueur impaire n'a **pas d'extension stable** et son extension grounded est **vide** (personne n'est défendable : chacun est attaqué par un argument non défait). Mais avec des valeurs, **l'audience décide quelles attaques réussissent**, ce qui brise la symétrie du cycle.

Construisons ce VAF et calculons ses extensions sous trois audiences distinctes.


In [3]:
// L'exemple canonique : cycle a->b->c->a, 3 valeurs distinctes.
var args = new[] { "a", "b", "c" };
var attacks = new[] { ("a", "b"), ("b", "c"), ("c", "a") };
var values = new Dictionary<string, string>
{
    ["a"] = "verite",
    ["b"] = "confiance",
    ["c"] = "solidarite"
};

var vaf = new VAF(args, attacks, values);
Console.WriteLine("VAF canonique:");
Console.WriteLine($"  {vaf}");
Console.WriteLine($"  Valeurs: [{string.Join(", ", vaf.ValueSet.OrderBy(s => s, StringComparer.Ordinal))}]");
Console.WriteLine();

// Trois audiences = trois ordres totaux differents, chacun preferant une
// valeur differente en tete.
var audiences = new (string Label, List<string> Audience)[]
{
    ("audience-1 (verite > confiance > solidarite)", new List<string> { "verite", "confiance", "solidarite" }),
    ("audience-2 (confiance > solidarite > verite)", new List<string> { "confiance", "solidarite", "verite" }),
    ("audience-3 (solidarite > verite > confiance)", new List<string> { "solidarite", "verite", "confiance" })
};

// Resultats attendus (issus du carnet Python jumeau) : affiches puis compares.
var attendu = new Dictionary<string, (string Attaques, string Grounded)>
{
    ["audience-1 (verite > confiance > solidarite)"] = ("[a->b, b->c]", "{a, c}"),
    ["audience-2 (confiance > solidarite > verite)"] = ("[b->c, c->a]", "{a, b}"),
    ["audience-3 (solidarite > verite > confiance)"] = ("[a->b, c->a]", "{b, c}")
};

Console.WriteLine("Extensions grounded sous chaque audience :");
Console.WriteLine(new string('-', 60));
foreach (var (label, aud) in audiences)
{
    var afAud = VafSemantics.AfFromAudience(vaf, aud);
    var g = DungSemantics.Grounded(afAud);
    var survived = afAud.Attacks
        .Select(t => t.Item1 + "->" + t.Item2)
        .OrderBy(s => s, StringComparer.Ordinal)
        .ToList();
    var survivedStr = "[" + string.Join(", ", survived) + "]";
    var gStr = AF.Format(g);
    var excluded = vaf.Args.Except(g).OrderBy(s => s, StringComparer.Ordinal);
    Console.WriteLine(label);
    Console.WriteLine($"    attaques reussissant: {survivedStr}");
    Console.WriteLine($"    grounded = {gStr}  (exclu: {string.Join(", ", excluded)})");
    var exp = attendu[label];
    Console.WriteLine($"    attendu: attaques {exp.Attaques}, grounded {exp.Grounded}");
    Checks.Require(survivedStr == exp.Attaques, $"{label} : attaques reussissant {survivedStr} (attendu {exp.Attaques})");
    Checks.Require(gStr == exp.Grounded, $"{label} : grounded {gStr} (attendu {exp.Grounded})");
    Console.WriteLine();
}

VAF canonique:


  VAF(args=[a, b, c], attacks=[a->b, b->c, c->a], values={a:verite, b:confiance, c:solidarite})


  Valeurs: [confiance, solidarite, verite]


Extensions grounded sous chaque audience :


------------------------------------------------------------


audience-1 (verite > confiance > solidarite)


... (13 outputs left out)

    grounded = {b, c}  (exclu: a)


    attendu: attaques [a->b, c->a], grounded {b, c}


  CONFORME : audience-3 (solidarite > verite > confiance) : attaques reussissant [a->b, c->a] (attendu [a->b, c->a])


  CONFORME : audience-3 (solidarite > verite > confiance) : grounded {b, c} (attendu {b, c})


## Ce que la divergence nous apprend

Les trois audiences produisent **trois extensions grounded différentes** sur le *même* graphe d'attaques :

| Audience (valeur en tête) | Attaques qui réussissent | Extension grounded |
|---------------------------|--------------------------|--------------------|
| `verite` en tête | $a \to b$, $b \to c$ (mais $c \not\to a$) | **{a, c}** (b exclu) |
| `confiance` en tête | $b \to c$, $c \to a$ (mais $a \not\to b$) | **{a, b}** (c exclu) |
| `solidarite` en tête | $c \to a$, $a \to b$ (mais $b \not\to c$) | **{b, c}** (a exclu) |

**Lecture** : sur ce cycle, le grounded contient toujours **deux** arguments sur trois — mais l'argument *exclu* change avec l'audience ($b$, puis $c$, puis $a$). Le mécanisme : l'argument dont la valeur est **la plus préférée** voit l'attaque dirigée contre lui échouer (son attaquant est moins préféré), donc il survit ; et comme sa propre attaque *vers le suivant* réussit, il élimine ce suivant. Le troisième argument survit alors par défense (le survivant principal défait l'attaquant du troisième). En résumé : **mettre une valeur en tête d'audience condamne l'argument que cette valeur attaque dans le cycle**.

C'est le résultat central de Bench-Capon : **la conclusion d'un débat n'est pas déterminée par le graphe d'arguments seul, mais par le graphe ET les valeurs de l'audience**. Trois auditoires de valeurs différentes, confrontés aux *mêmes* arguments, excluent légitimement des arguments différents. Cela relie l'argumentation à la théorie du choix social : si plusieurs individus ont des audiences différentes, **agréger leurs préférences** (cf. [Tweety-9](../Tweety/Tweety-09-Preferences-Python.ipynb), [GameTheory SocialChoice](../../GameTheory/)) devient la question centrale pour décider collectivement.


## Sweep d'audiences : combien de conclusions distinctes ?

Sur un VAF à $|V|$ valeurs, il y a $|V|!$ audiences possibles (tous les ordres totaux). Une question naturelle : parmi toutes ces audiences, **combien d'extensions grounded distinctes** obtient-on ? Si la réponse est 1, les valeurs sont décoratives (elles ne changent rien). Si la réponse est grande, le débat est *sensible* à l'audience — ce qui est précisément la situation où l'argumentation basée sur les valeurs apporte quelque chose que Dung pur ne peut pas exprimer.

Énumérons toutes les audiences sur notre VAF à 3 valeurs et comptons les outcomes.


In [4]:
// Sweep : toutes les audiences (ordres totaux sur les valeurs).
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Enumeration deterministe des ordres totaux (remplace itertools.permutations).
IEnumerable<List<string>> Permutations(IList<string> items)
{
    if (items.Count <= 1)
    {
        yield return new List<string>(items);
        yield break;
    }
    for (int i = 0; i < items.Count; i++)
    {
        var rest = items.Where((_, j) => j != i).ToList();
        foreach (var tail in Permutations(rest))
        {
            var perm = new List<string> { items[i] };
            perm.AddRange(tail);
            yield return perm;
        }
    }
}

// Denombrement exact du nombre d'ordres totaux : n! en BigInteger.
BigInteger Factorial(int n)
{
    BigInteger result = BigInteger.One;
    for (int k = 2; k <= n; k++)
        result *= k;
    return result;
}

var sortedValues = vaf.ValueSet.OrderBy(s => s, StringComparer.Ordinal).ToList();
Console.WriteLine("Sweep complet des audiences sur le VAF canonique (3! = 6 ordres):");
Console.WriteLine(new string('-', 65));

var distinctOutcomes = new Dictionary<string, List<List<string>>>();
var resultats = new List<(List<string> Audience, string Grounded)>();
foreach (var aud in Permutations(sortedValues))
{
    var afAud = VafSemantics.AfFromAudience(vaf, aud);
    var g = DungSemantics.Grounded(afAud);
    var key = g.Count > 0
        ? string.Join(", ", g.OrderBy(s => s, StringComparer.Ordinal))
        : "()";
    if (!distinctOutcomes.TryGetValue(key, out var bucket))
    {
        bucket = new List<List<string>>();
        distinctOutcomes[key] = bucket;
    }
    bucket.Add(aud);

    var gStr = g.Count > 0 ? "{" + string.Join(", ", g.OrderBy(s => s, StringComparer.Ordinal)) + "}" : "{}";
    resultats.Add((aud, gStr));
    Console.WriteLine($"  audience [{string.Join(", ", aud)}] -> grounded {gStr}");
}

BigInteger audienceCount = resultats.Count;
BigInteger expectedAudienceCount = Factorial(sortedValues.Count);
BigInteger distinctOutcomeCount = distinctOutcomes.Count;

Console.WriteLine(new string('-', 65));
Console.WriteLine($"Distinct grounded outcomes: {distinctOutcomeCount.ToString(CultureInfo.InvariantCulture)}");
foreach (var outcome in distinctOutcomes.OrderBy(kv => kv.Key, StringComparer.Ordinal))
{
    BigInteger audienceOutcomeCount = outcome.Value.Count;
    Console.WriteLine($"  ({outcome.Key}) atteint par {audienceOutcomeCount.ToString(CultureInfo.InvariantCulture)} audience(s)");
}

// --- Assertions executables : repartition des 6 audiences ---
Checks.Require(audienceCount == expectedAudienceCount, "6 audiences enumerees sur les 3 valeurs (3! = 6)");
Checks.Require(distinctOutcomeCount == new BigInteger(3), "3 conclusions grounded distinctes");
Checks.Require(distinctOutcomes.Values.All(v => new BigInteger(v.Count) == new BigInteger(2)), "chaque conclusion est portee par exactement 2 audiences");
var attenduParTete = new Dictionary<string, string>
{
    ["verite"] = "{a, c}",
    ["confiance"] = "{a, b}",
    ["solidarite"] = "{b, c}"
};
foreach (var (aud, gStr) in resultats)
{
    var tete = aud[0];
    attenduParTete.TryGetValue(tete, out var attenduG);
    Checks.Require(attenduG == gStr, $"audience menee par {tete} -> grounded {gStr} (attendu {attenduG})");
}

Sweep complet des audiences sur le VAF canonique (3! = 6 ordres):


-----------------------------------------------------------------


  audience [confiance, solidarite, verite] -> grounded {a, b}


  audience [confiance, verite, solidarite] -> grounded {a, b}


  audience [solidarite, confiance, verite] -> grounded {b, c}


  audience [solidarite, verite, confiance] -> grounded {b, c}


... (12 outputs left out)

  CONFORME : audience menee par solidarite -> grounded {b, c} (attendu {b, c})


  CONFORME : audience menee par solidarite -> grounded {b, c} (attendu {b, c})


  CONFORME : audience menee par verite -> grounded {a, c} (attendu {a, c})


  CONFORME : audience menee par verite -> grounded {a, c} (attendu {a, c})


**Résultat** : sur 6 audiences possibles, on obtient **3 conclusions distinctes** — {a, b}, {a, c}, {b, c} — chacune atteinte par **2** audiences. (Les deux audiences qui mettent la même valeur en tête convergent vers la même extension : seul compte quel argument est « en tête ».) Le débat est donc *fortement sensible* à l'audience : il n'y a pas de verdict universel, la conclusion dépend de qui écoute.

C'est un diagnostic actionnable : si un médiateur veut faire converger un débat argumentatif bloqué en cycle, il doit d'abord **révéler les valeurs** des parties et chercher un compromis sur l'ordre de préférence — pas seulement sur les arguments eux-mêmes.


## Vérifiez votre compréhension

Les trois exercices suivants approfondissent le cadre. Chaque stub est à compléter — le notebook s'exécute de bout en bout même non complété (les exercices renvoient une valeur neutre — `null` ou un dictionnaire vide — et affichent un message). La méthode de chaque exercice est déjà démontrée sur un exemple dans le carnet :

- **Exercice 1** (choisir une audience pour rejeter un argument donné) → la section *Exemple canonique* l'a fait dans les trois sens : l'audience-1 exclut `b` (`grounded = {a, c}`), l'audience-2 exclut `c`, l'audience-3 exclut `a`.
- **Exercice 2** (quand le VAF dégénère en Dung pur) → le smoke test initial l'a mesuré : valeurs égales `['v', 'v']` → `attaque reussit` → `grounded = {a}`, exactement le comportement du cadre de Dung.
- **Exercice 3** (cartographier toutes les conclusions d'un VAF plus large) → le sweep de la section *Sweep d'audiences* l'a fait sur 6 audiences : `Distinct grounded outcomes: 3`, chaque issue portée par deux audiences.


### Exercice 1 — Choisir une audience pour rejeter un argument donné

On a vu que chaque audience exclut un argument différent du grounded. **Objectif** : écrire `AudienceQuiRejette(vaf, cible)` qui renvoie *une* audience rendant `cible` **absent** du grounded (rejeté).

Indice : d'après la lecture ci-dessus, l'argument exclu est celui **attaqué par l'argument dont la valeur est la plus préférée**. Donc, pour rejeter `cible`, il faut mettre en tête d'audience la valeur de son *attaquant* dans le graphe.


In [5]:
// Exercice 1 : a completer
// TODO etudiant : trouver x tel que (x, cible) est une attaque du VAF,
// puis construire une audience dont la premiere valeur est celle de x.
List<string> AudienceQuiRejette(VAF vaf, string cible)
{
    // TODO etudiant
    return null;
}

// Verification : une solution fournie est reellement controlee ; si elle
// n'exclut pas la cible, l'execution s'interrompt (ECHEC).
var audEx1 = AudienceQuiRejette(vaf, "b");
if (audEx1 == null)
{
    Console.WriteLine("Exercice 1 a completer.");
}
else
{
    try
    {
        // L'audience doit etre une permutation complete des valeurs : une liste
        // vide ou incomplete ne peut pas servir de solution.
        VafSemantics.ValidateAudience(vaf, audEx1);
        var afAudEx1 = VafSemantics.AfFromAudience(vaf, audEx1);
        var gEx1 = DungSemantics.Grounded(afAudEx1);
        Console.WriteLine($"Audience trouvee: [{string.Join(", ", audEx1)}] -> grounded = {AF.Format(gEx1)}");
        Checks.Require(!gEx1.Contains("b"), "l'audience fournie exclut b du grounded");
        Console.WriteLine("SUCCES : b est rejete sous cette audience.");
    }
    catch (ArgumentException ex)
    {
        Checks.Require(false, ex.Message);
    }
}


Exercice 1 a completer.


### Exercice 2 — Quand le VAF dégénère en Dung pur

Sur le VAF canonique, les valeurs sont distinctes et l'audience brise le cycle. Mais que se passe-t-il si **tous les arguments partagent la même valeur** ? Alors toute attaque a un attaquant et une cible de valeur égale → toutes les attaques réussissent, et l'on retrouve exactement le cycle de Dung pur : le grounded est **vide**.

**Objectif** : construire un VAF (cycle $a \to b \to c \to a$) où tous les arguments ont la même valeur, choisir une audience triviale, et vérifier que le grounded est bien vide — confirmant que le VAF *contient* Dung comme cas particulier (valeurs dégénérées).


In [6]:
// Exercice 2 : a completer
// TODO etudiant : construire le cycle a->b->c->a avec une valeur unique pour
// les trois arguments, et une audience triviale ne contenant que cette valeur.
(VAF, List<string>) VafDegenererEnDung()
{
    // TODO etudiant
    return (null, null);
}

// Verification : une solution fournie est reellement controlee ; si le cycle
// de Dung attendu n'est pas obtenu, l'execution s'interrompt (ECHEC).
var (vafEx2, audEx2) = VafDegenererEnDung();
if (vafEx2 == null || audEx2 == null)
{
    Console.WriteLine("Exercice 2 a completer.");
}
else
{
    var afEx2 = VafSemantics.AfFromAudience(vafEx2, audEx2);
    var gEx2 = DungSemantics.Grounded(afEx2);
    Console.WriteLine($"Attaques reussissant: {AF.FormatAttacks(afEx2.Attacks)}");
    Console.WriteLine($"Grounded: {(gEx2.Count > 0 ? AF.Format(gEx2) : "{} (vide)")}");
    Checks.Require(vafEx2.Args.SetEquals(new[] { "a", "b", "c" }),
        "le VAF degenere contient exactement les arguments a, b et c");
    Checks.Require(vafEx2.ValueSet.Count == 1,
        "les trois arguments partagent une seule valeur commune");
    Checks.Require(vafEx2.Attacks.SetEquals(new[] { ("a", "b"), ("b", "c"), ("c", "a") }),
        "la relation d'attaque est exactement le cycle a->b->c->a");
    Checks.Require(afEx2.Attacks.Count == 3, "les trois attaques du cycle reussissent (valeurs identiques)");
    Checks.Require(gEx2.Count == 0, "le grounded du cycle impair de Dung est vide");
    Console.WriteLine("SUCCES : VAF degenere en Dung, grounded vide.");
}


Exercice 2 a completer.


### Exercice 3 — Cartographier toutes les conclusions d'un VAF plus large

Généralisez le sweep : écrivez `ToutesLesConclusions(vaf)` qui renvoie un dictionnaire `{extension_grounded: [liste des audiences qui la produisent]}` pour **toutes** les audiences possibles.

**Objectif** : appliquer à un VAF à **4 arguments** et **3 valeurs** (certains arguments partageant une valeur), et interpréter pourquoi le nombre de conclusions distinctes est (généralement) inférieur au nombre d'audiences.


In [7]:
// Exercice 3 : a completer
using System;
using System.Collections.Generic;
using System.Linq;

// TODO etudiant : enumerer tous les ordres totaux sur les valeurs du VAF,
// calculer l'extension grounded de chacun via AfFromAudience + Grounded,
// puis regrouper les audiences par outcome (cle = extension grounded).
Dictionary<string, List<List<string>>> ToutesLesConclusions(VAF vaf)
{
    // TODO etudiant
    return new Dictionary<string, List<List<string>>>();
}

// Enumeration deterministe des ordres totaux (helper de verification).
IEnumerable<List<string>> PermutationsAll(IList<string> items)
{
    if (items.Count <= 1)
    {
        yield return new List<string>(items);
        yield break;
    }
    for (int i = 0; i < items.Count; i++)
    {
        var rest = items.Where((_, j) => j != i).ToList();
        foreach (var tail in PermutationsAll(rest))
        {
            var perm = new List<string> { items[i] };
            perm.AddRange(tail);
            yield return perm;
        }
    }
}

// Verification : activee apres completion de la fonction ci-dessus.
// - VAF canonique (3 valeurs) : doit reproduire le sweep mesure en cellule 8.
// - VAF large (4 arguments, 3 valeurs : a et d partagent 'verite') : les
//   conclusions sont calculees par la fonction de l'etudiant, puis verifiees
//   structurellement ; aucune valeur de reponse n'est fournie ici.
var conclu = ToutesLesConclusions(vaf);
if (conclu.Count == 0)
{
    Console.WriteLine("Exercice 3 a completer.");
}
else
{
    Console.WriteLine($"Le VAF canonique a {conclu.Count} conclusion(s) distincte(s):");
    foreach (var kv in conclu.OrderBy(k => k.Key, StringComparer.Ordinal))
    {
        var lab = kv.Key.Length > 0 ? "{" + kv.Key + "}" : "{} (vide)";
        Console.WriteLine($"  {lab} <- {kv.Value.Count} audience(s)");
    }

    var vafLarge = new VAF(new[] { "a", "b", "c", "d" },
        new[] { ("a", "b"), ("b", "c"), ("c", "d"), ("d", "a") },
        new Dictionary<string, string>
        {
            ["a"] = "verite",
            ["b"] = "confiance",
            ["c"] = "solidarite",
            ["d"] = "verite"
        });
    var concluLarge = ToutesLesConclusions(vafLarge);
    Console.WriteLine($"Le VAF large (4 args, 3 valeurs) a {concluLarge.Count} conclusion(s) distincte(s):");
    foreach (var kv in concluLarge.OrderBy(k => k.Key, StringComparer.Ordinal))
    {
        var lab = kv.Key.Length > 0 ? "{" + kv.Key + "}" : "{} (vide)";
        Console.WriteLine($"  {lab} <- {kv.Value.Count} audience(s)");
    }

    // Controles structurels : les deux VAF doivent voir chaque ordre total
    // apparaitre exactement une fois, et aucun groupe ne doit etre vide.
    var expectedAudiences = PermutationsAll(
        vaf.ValueSet.OrderBy(v => v, StringComparer.Ordinal).ToList())
        .Select(a => string.Join(",", a))
        .OrderBy(a => a, StringComparer.Ordinal)
        .ToList();
    var canonicalAudiences = conclu.Values.SelectMany(v => v)
        .Select(a => string.Join(",", a))
        .OrderBy(a => a, StringComparer.Ordinal)
        .ToList();

    Checks.Require(conclu.Values.All(v => v.Count > 0),
        "chaque groupe de conclusions canoniques contient au moins une audience");
    Checks.Require(canonicalAudiences.Count == expectedAudiences.Count,
        "chaque ordre canonique apparait exactement une fois");
    Checks.Require(canonicalAudiences.Distinct(StringComparer.Ordinal).Count() == canonicalAudiences.Count,
        "les audiences canoniques sont toutes distinctes");
    Checks.Require(canonicalAudiences.SequenceEqual(expectedAudiences, StringComparer.Ordinal),
        "les audiences canoniques couvrent exactement tous les ordres totaux");

    var expectedAudiencesLarge = PermutationsAll(
        vafLarge.ValueSet.OrderBy(v => v, StringComparer.Ordinal).ToList())
        .Select(a => string.Join(",", a))
        .OrderBy(a => a, StringComparer.Ordinal)
        .ToList();
    var largeAudiences = concluLarge.Values.SelectMany(v => v)
        .Select(a => string.Join(",", a))
        .OrderBy(a => a, StringComparer.Ordinal)
        .ToList();

    Checks.Require(concluLarge.Values.All(v => v.Count > 0),
        "chaque groupe de conclusions du VAF large contient au moins une audience");
    Checks.Require(largeAudiences.Count == expectedAudiencesLarge.Count,
        "chaque ordre du VAF large apparait exactement une fois");
    Checks.Require(largeAudiences.Distinct(StringComparer.Ordinal).Count() == largeAudiences.Count,
        "les audiences du VAF large sont toutes distinctes");
    Checks.Require(largeAudiences.SequenceEqual(expectedAudiencesLarge, StringComparer.Ordinal),
        "les audiences du VAF large couvrent exactement tous les ordres totaux");

    // Pour chaque groupe, on recalcule le grounded de chaque audience et on
    // compare ce resultat a la cle fournie : une cle arbitraire est rejetee.
    foreach (var group in concluLarge)
    {
        foreach (var audience in group.Value)
        {
            var recalculated = DungSemantics.Grounded(VafSemantics.AfFromAudience(vafLarge, audience))
                .OrderBy(s => s, StringComparer.Ordinal);
            var recalculatedKey = string.Join(", ", recalculated);
            Checks.Require(recalculatedKey == group.Key,
                "chaque audience du VAF large est rattachee a sa vraie extension grounded");
        }
    }

    // Coherence avec le sweep canonique deja mesure en cellule 8.
    Checks.Require(conclu.Count == 3, "le VAF canonique a 3 conclusions distinctes (sweep de la cellule 8)");
    Checks.Require(conclu.Values.All(v => v.Count == 2), "chaque conclusion canonique est portee par 2 audiences");
    foreach (var group in conclu)
    {
        foreach (var audience in group.Value)
        {
            var recalculated = DungSemantics.Grounded(VafSemantics.AfFromAudience(vaf, audience))
                .OrderBy(s => s, StringComparer.Ordinal);
            var recalculatedKey = string.Join(", ", recalculated);
            Checks.Require(recalculatedKey == group.Key,
                "chaque audience canonique est rattachee a sa vraie extension grounded");
        }
    }
}


Exercice 3 a completer.


## À retenir

Le cadre de Dung (1995) dit *qui attaque qui*. Le cadre de Bench-Capon (2003) ajoute une couche critique : **qui défait qui dépend de ce que valorise l'audience**. Concrètement :

- Une **attaque** $x \to y$ n'est pas automatiquement une **défaite** : elle ne réussit que si la valeur promue par $x$ est au moins aussi préférée (pour l'audience) que celle promue par $y$. Sur le cycle canonique, les trois audiences nommées en tête d'énoncé (vérité, confiance, solidarité) enregistrent chacune **deux** attaques réussissantes sur trois — la troisième échoue précisément parce que sa valeur est moins bien classée. Attention : ce décompte vaut pour ces trois-là uniquement ; d'autres ordres totaux, comme `confiance > verite > solidarite`, n'en conservent qu'**une**.
- Changer l'audience change la relation de défaite, donc change les extensions : l'audience-1 (vérité > confiance > solidarité) rend `grounded = {a, c}` et exclut `b` ; l'audience-2 exclut `c` ; l'audience-3 exclut `a`. Trois audiences excluent trois arguments différents — la signature d'un débat **sensible à l'audience**.
- Le sweep des 6 audiences ne produit que `3` issues distinctes, chaque issue portée par deux audiences : sur ce cycle canonique, le verdict ne dépend que de la **valeur en tête** (les deux audiences menées par `confiance` rendent `{a, b}`, celles menées par `solidarite` rendent `{b, c}`, celles menées par `verite` rendent `{a, c}`) — la permutation des valeurs suivantes ne change rien.
- Cas limite : si tous les arguments partagent une même valeur, le VAF dégénère en cadre de Dung pur — mesuré au smoke test : `valeurs egales -> attaque reussit -> a defait b`. Le cadre de Dung est donc un cas particulier du VAF.

## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place du cadre VAF dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet, du raisonnement formel aux systèmes multi-agents.

**Agrégation d'audiences → choix social** : si chaque individu a sa propre audience, comment décider collectivement ? C'est le pont vers [Tweety-9 (préférences)](../Tweety/Tweety-09-Preferences-Python.ipynb) et [GameTheory / SocialChoice](../../GameTheory/) (théorèmes d'impossibilité d'Arrow, votations). La question « existe-t-il une agrégation d'audiences qui préserve la rationalité ? » est un analogue direct du paradoxe de Condorcet.

| [jumeau Rust](Argumentation-03b-Value-Based-AF-Rust.ipynb) | [jumeau Python](Argumentation-03b-Value-Based-AF-Python.ipynb)

**Argumentation graduée** : plutôt qu'un ordre total sur les valeurs, [Argumentation-03c-Ranking-Semantics-Python](Argumentation-03c-Ranking-Semantics-Python.ipynb) attribue une *force numérique* à chaque argument — une autre façon de départager ce que Dung laisse indécis.

**Bipolarité** : les VAF gèrent les préférences, mais restent *attack-only*. Les cadres bipolaires (Cayrol & Lagasquie-Schiex, 2005) ajoutent une relation de **support** — un prolongement naturel au-delà de ce notebook.

**Référence** — Trevor J. M. Bench-Capon, *« A Value-Based Argumentation Framework »*, dans *Proceedings of the European Conference on Logics in Artificial Intelligence (JELIA)*, 2003, le papier fondateur du cadre VAF reconstruit ici.

***

## Attestation de parité (C# / Python)

Ce jumeau rejoue le carnet [Argumentation-03b-Value-Based-AF-Python](Argumentation-03b-Value-Based-AF-Python.ipynb) à l'identique : mêmes entrées, mêmes sorties mesurées, aucun générateur aléatoire.

| Mesure | Python | C# (ce carnet) |
|--------|--------|-----------------|
| Primitives Dung : `grounded` du cadre `{a, b}`, attaque `a->b` | `{a}` | `{a}` |
| VAF minimal (valeur unique `v`) : attaques réussissantes | `{('a', 'b')}` | `{a->b}` |
| Cycle canonique, audience-1 (verite > confiance > solidarite) | grounded `{a, c}`, `b` exclu | grounded `{a, c}`, `b` exclu |
| Cycle canonique, audience-2 (confiance > solidarite > verite) | grounded `{a, b}`, `c` exclu | grounded `{a, b}`, `c` exclu |
| Cycle canonique, audience-3 (solidarite > verite > confiance) | grounded `{b, c}`, `a` exclu | grounded `{b, c}`, `a` exclu |
| Sweep des 6 audiences : conclusions distinctes | `3` | `3` |
| Sweep : nombre d'audiences par conclusion | `2` | `2` |

Toutes les valeurs ci-dessus sont **recalculées** par les cellules de ce carnet sur le noyau `.net-csharp` (BCL .NET seule : `System.Collections.Generic`, `System.Linq`) ; aucune n'est recopiée d'une sortie éditée à la main.

***

*Notebook pédagogique — C# pur (BCL .NET, sans NuGet ni réseau), déterministe, sans LLM ni solveur externe. Conforme C.1 (stubs sans erreur volontaire), C.2 (sorties réelles), exercices à compléter.*